[Guide Index](README.md)


# 02.08 OWL 2 Manchester Syntax

Manchester Syntax is an English-readable notation for OWL 2 ontologies — `Class: Cat SubClassOf: Animal` reads more naturally, rather than as raw RDF triples:

| Manchester Syntax | Equivalent RDF (Turtle) |
|---|---|
| `Class: Cat SubClassOf: Animal` | `:Cat rdf:type owl:Class . :Cat rdfs:subClassOf :Animal .` |

`StarLayer` is able to parse Manchester text into its RDF equivalent, and serialize an RDF graph back to Manchester text. `StarLayerGraph` operates over RDF graphs, so an ontology expressed as Manchester Syntax needs to be translated into a graph before it can be used with `StarLayerGraph`; for example, as an ontology to support OWL-based reasoning (see 02.06 and 02.07).

This guide starts with Manchester Syntax to RDF (section 02.08.1), and RDF back to Manchester Syntax. The rest of the guide goes through the notation construct by construct, each with its RDF equivalent.  

A Manchester Syntax document's own structure - frames, clauses, expressions - can also be treated as an inspectable, editable RDF graph. That is covered in the [OWL Manchester Syntax as RDF guide](06.05-manchester-syntax-as-rdf.ipynb).


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). No extra runtime dependencies are needed - Manchester Syntax parsing is pure Python.

Run cells from top to bottom; later cells reuse variables from earlier ones.

A `StarLayerGraph` starts with no prefix bindings, so the setup cell sets default prefixes (`:`, `owl:`, `rdfs:`) that every new graph starts with; `serialize(format="turtle", pretty=True)` then writes grouped, nested Turtle with only the prefixes that are used. (`xsd:` is added by the Manchester parser when a document uses it.)


In [1]:
from starlayer.graph import Namespace, OWL, RDFS, StarLayerGraph, set_default_prefixes

EX = Namespace("http://example.org/vehicles#")

# every graph made after this call starts with these prefixes bound
set_default_prefixes({"": EX, "owl": OWL, "rdfs": RDFS})


def show(manchester_text):
    """Parse a Manchester Syntax snippet and print the RDF it maps to."""
    g = StarLayerGraph()
    g.parse(data=f"Prefix: : <{EX}>\n" + manchester_text, format="manchester")
    print(g.serialize(format="turtle", pretty=True))
    return g

## 02.08.1 Manchester Syntax to RDF

`g.parse(data=..., format="manchester")` reads Manchester Syntax text into a graph as OWL RDF. Each frame (`Class:`, `ObjectProperty:`, `Individual:` etc) becomes the triples for the axioms in its clauses. A `Prefix:` line gives the text its names using the default prefix `:`. 

Text that is not Manchester Syntax raises a `ManchesterSyntaxError`. Some pythons tools that accept Manchester Syntax (e.g. owl api) also accept SWRL rules. StarLayer does not support SWRL rules and text that includes rule frames are rejected.

Below, a small ontology is parsed into a graph and the RDF is printed as Turtle.

In [ ]:
ontology = f"Prefix: : <{EX}>\n" + """
Class: Animal
Class: Leg
Class: Cat
    SubClassOf: Animal
    SubClassOf: hasLeg some Leg
    Annotations: rdfs:label "Cat"
ObjectProperty: hasLeg
"""

g = StarLayerGraph()
g.parse(data=ontology, format="manchester")
print(g.serialize(format="turtle", pretty=True))

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Animal a owl:Class .

:Cat a owl:Class ;
    rdfs:label "Cat" ;
    rdfs:subClassOf :Animal, [
        a owl:Restriction ;
        owl:onProperty :hasLeg ;
        owl:someValuesFrom :Leg
    ] .

:Leg a owl:Class .

:hasLeg a owl:ObjectProperty .



## 02.08.2 RDF to Manchester Syntax

`g.serialize(format="manchester")` writes a graph as Manchester Syntax text, the reverse direction.  Some axioms are returned in their canonical expanded form so they may not match the orginal input identically. However, the precise meaning is retained. 

The graph is ordinary RDF, so it can be edited with standard graph operations before it is written back.

Below, the graph from section 02.08.1 is written as Manchester Syntax. Then an edit is written as Turtle, parsed into a small graph, and added to `g`: a cat has some owner who is a person. The graph is written again, and the new axiom appears in the text.

In [13]:
print(g.serialize(format="manchester"))

# the edit, written as Turtle: Cat SubClassOf hasOwner some Person
edit = StarLayerGraph()
edit.parse(data="""
@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Person a owl:Class .
:hasOwner a owl:ObjectProperty .
:Cat rdfs:subClassOf [
    a owl:Restriction ;
    owl:onProperty :hasOwner ;
    owl:someValuesFrom :Person
] .
""", format="turtle")

# add the edit's triples to g
g += edit

print("after the edit:")
print(g.serialize(format="manchester"))

Prefix: : <http://example.org/vehicles#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>

Class: :Animal

Class: :Cat
    SubClassOf: :hasLeg some :Leg, :Animal
    Annotations: rdfs:label "Cat"

Class: :Leg

ObjectProperty: :hasLeg

after the edit:
Prefix: : <http://example.org/vehicles#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>

Class: :Animal

Class: :Cat
    SubClassOf: :hasOwner some :Person, :hasLeg some :Leg, :Animal
    Annotations: rdfs:label "Cat"

Class: :Leg

Class: :Person

ObjectProperty: :hasLeg

ObjectProperty: :hasOwner



## 02.08.3 Frames

An ontology in Manchester Syntax is a list of frames. Each frame names one entity (a class, a property or an individual) and lists clauses that say things about it. The sections below take various frame types in turn. Each section shows Manchester Syntax text and the RDF it produces, using the `show()` helper defined above.

Each clause is an *axiom*: a statement the ontology asserts to be true. A reasoner uses axioms to draw conclusions, for example that an individual belongs to a class; the conclusions appear only after inference (see [02.06](02.06-graphs-inferencing.ipynb) and [02.07](02.07-owl-dl-reasoning.ipynb)). Axioms do not reject data that breaks them. OWL also does not assume that what is not stated is false (the open world assumption): a car with no stated owner is not an error, it has an owner that is not known. To reject data that breaks a rule, use SHACL (see [05.01](05.01-validating-with-shacl-1.2.ipynb)).

### 02.08.3.1 Classes: `SubClassOf:`, `SuperClassOf:`, `DisjointWith:`

A class is a set of individuals. Each `Class:` frame declares a class (`owl:Class`), and its clauses add axioms about it:

- `SubClassOf: Vehicle` means every `Car` is a `Vehicle`. An individual of type `Car` is also an individual of type `Vehicle`, once a reasoner has drawn that conclusion.
- `SuperClassOf:` is the reverse spelling of `SubClassOf:`. `Vehicle SuperClassOf: Motorcycle` means the same thing as `Motorcycle SubClassOf: Vehicle`.
- `DisjointWith: Car` means nothing can be both a `Truck` and a `Car`. If an individual is stated to be both, the ontology is inconsistent and a DL reasoner reports it.

The parser writes only the triples the text states. What follows from them, such as inheritance, is the work of a reasoner.

Below, a small class hierarchy is parsed and the RDF is printed.

In [2]:
_ = show("""
Class: Vehicle
Class: Car
    SubClassOf: Vehicle
Class: Motorcycle
    SubClassOf: Vehicle
Class: Truck
    SubClassOf: Vehicle
    DisjointWith: Car

Class: Vehicle
    SuperClassOf: Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Motorcycle a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Truck a owl:Class ;
    rdfs:subClassOf :Vehicle ;
    owl:disjointWith :Car .

:Vehicle a owl:Class .



### 02.08.3.2 Object and data properties

A property relates individuals. An object property relates an individual to another individual (`hasEngine`), and a data property relates an individual to a literal value (`hasWeightKg`). The clauses of a property frame mean:

- `Domain: Vehicle` means anything that has this property is concluded to be a `Vehicle`. It is not a check: using `hasEngine` on something makes it a `Vehicle`.
- `Range: Engine` means the values of the property are concluded to be `Engine`s. For a data property the range is a datatype, such as `xsd:decimal`.
- `Characteristics: Functional` means each individual has at most one value for the property. Other characteristics are `InverseFunctional`, `Transitive`, `Symmetric`, `Asymmetric`, `Reflexive` and `Irreflexive`.
- `InverseOf: hasEngine` means `poweredBy` runs the other way: if A `hasEngine` B, then B `poweredBy` A.
- `SubPropertyOf: hasEngine` means whenever two individuals are linked by `hasSubEngine`, they are also linked by `hasEngine`. `SuperPropertyOf:` is the reverse spelling.
- `SubPropertyChain: hasParent o hasParent` means that if A `hasParent` B and B `hasParent` C, then A `hasGrandparent` C. It applies to object properties only.

Below, each of these is parsed and the RDF is printed.

In [6]:
_ = show("""
Class: Vehicle
Class: Engine
ObjectProperty: hasEngine
    Domain: Vehicle
    Range: Engine
    Characteristics: Functional

ObjectProperty: poweredBy
    InverseOf: hasEngine

ObjectProperty: hasSubEngine
    SubPropertyOf: hasEngine

ObjectProperty: hasComponent
    SuperPropertyOf: hasSubEngine

ObjectProperty: hasParent
ObjectProperty: hasGrandparent
    SubPropertyChain: hasParent o hasParent

DataProperty: hasWeightKg
    Domain: Vehicle
    Range: xsd:decimal
    Characteristics: Functional
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:Engine a owl:Class .

:Vehicle a owl:Class .

:hasComponent a owl:ObjectProperty .

:hasEngine a owl:FunctionalProperty, owl:ObjectProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range :Engine .

:hasGrandparent a owl:ObjectProperty ;
    owl:propertyChainAxiom ( :hasParent :hasParent ) .

:hasParent a owl:ObjectProperty .

:hasSubEngine a owl:ObjectProperty ;
    rdfs:subPropertyOf :hasComponent, :hasEngine .

:hasWeightKg a owl:DatatypeProperty, owl:FunctionalProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range xsd:decimal .

:poweredBy a owl:ObjectProperty ;
    owl:inverseOf :hasEngine .



### 02.08.3.3 Individuals: `Types:`, `Facts:`, `SameAs:`, `DifferentFrom:`

An individual is a particular thing. Its frame says what it is and what is known about it:

- `Types: Car` means the individual is a member of the class `Car`.
- `Facts: hasOwner Alice` states a fact about the individual: a link to another individual, or a data value such as `hasVIN "..."`. A leading `not` states that a fact is false (`owl:NegativePropertyAssertion`), so `not hasOwner Bob` says Bob is not the owner. It is not silently dropped.
- `SameAs: mycar2` means the two names refer to the same individual.
- `DifferentFrom: Bob` means the two are different individuals.

OWL does not assume that different names mean different individuals (no unique name assumption). Two names are only known to be different when the ontology says so, with `DifferentFrom:` or `DifferentIndividuals:`.

Below, an individual with a type, facts, a `SameAs:` and a `DifferentFrom:` is parsed and the RDF is printed.

In [7]:
_ = show("""
Class: Car
ObjectProperty: hasOwner
DataProperty: hasVIN
Individual: Alice
Individual: Bob
Individual: mycar2

Individual: mycar
    Types: Car
    Facts: hasOwner Alice, hasVIN "1HGCM82633A004352", not hasOwner Bob
    SameAs: mycar2
    DifferentFrom: Bob
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Alice a owl:NamedIndividual .

:Bob a owl:NamedIndividual .

:Car a owl:Class .

:hasOwner a owl:ObjectProperty .

:hasVIN a owl:DatatypeProperty .

:mycar a :Car, owl:NamedIndividual ;
    :hasOwner :Alice ;
    :hasVIN "1HGCM82633A004352" ;
    owl:differentFrom :Bob ;
    owl:sameAs :mycar2 .

:mycar2 a owl:NamedIndividual .

_:b0 a owl:NegativePropertyAssertion ;
    owl:assertionProperty :hasOwner ;
    owl:sourceIndividual :mycar ;
    owl:targetIndividual :Bob .



### 02.08.3.4 `HasKey:` and `DisjointUnionOf:`

Two more axioms about a class:

- `HasKey: hasVIN` means a property (or a set of properties) identifies the named individuals of the class: two named individuals of the class with the same value for every key property are concluded to be the same individual. It is a conclusion a reasoner draws, not a uniqueness check on the data.
- `DisjointUnionOf: Car, Truck, Motorcycle` means a `Vehicle` is exactly a `Car`, a `Truck` or a `Motorcycle`, and nothing is in more than one of them. The subclasses are disjoint and together make up the class.

Below, a class with both axioms is parsed and the RDF is printed.

In [9]:
_ = show("""
Class: Car
Class: Truck
Class: Motorcycle
DataProperty: hasVIN

Class: Vehicle
    HasKey: hasVIN
    DisjointUnionOf: Car, Truck, Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class .

:Motorcycle a owl:Class .

:Truck a owl:Class .

:Vehicle a owl:Class ;
    owl:disjointUnionOf ( :Car :Truck :Motorcycle ) ;
    owl:hasKey ( :hasVIN ) .

:hasVIN a owl:DatatypeProperty .



## 02.08.4 Expressions

Clauses refer to classes, properties and data ranges, and can describe new ones with expressions: combinations with `and`, `or` and `not`, restrictions on a property, enumerations, and datatypes restricted by facets. The sections below take each kind of expression in turn.

An expression describes a class (or a data range) by its conditions: the set of all individuals that meet them. It can be used wherever a class name can, for example after `SubClassOf:` or `EquivalentTo:`.

### 02.08.4.1 Class expressions: `and` / `or` / `not` / `that`

Class expressions combine classes with set operations:

- `Car and Truck` is the individuals in both classes (the intersection). `that` is a synonym for `and`, used to refine a description in a sentence-like way ("a family vehicle that has airbags").
- `Car or Truck` is the individuals in either class (the union).
- `not Truck` is every individual that is not a `Truck` (the complement).

In the example, `ElectricCar SubClassOf: Car and (not Truck)` means every `ElectricCar` is a `Car` and not a `Truck`. `FamilyVehicle EquivalentTo: Car or Truck` is a definition in both directions: an individual is a `FamilyVehicle` if and only if it is a `Car` or a `Truck`. `SubClassOf:` works in one direction only, so an `EquivalentTo:` definition lets a reasoner place individuals in the class as well as draw conclusions from membership.

Below, these expressions are parsed and the RDF is printed.

In [3]:
_ = show("""
Class: Car
Class: Truck
DataProperty: hasAirbags

Class: ElectricCar
    SubClassOf: Car and (not Truck)

Class: FamilyVehicle
    EquivalentTo: Car or Truck

Class: SafeFamilyVehicle
    EquivalentTo: FamilyVehicle that (hasAirbags value true)
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class .

:ElectricCar a owl:Class ;
    rdfs:subClassOf [
        a owl:Class ;
        owl:intersectionOf (
            :Car
            [
                a owl:Class ;
                owl:complementOf :Truck
            ]
        )
    ] .

:FamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:unionOf ( :Car :Truck )
    ] .

:SafeFamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:intersectionOf (
            :FamilyVehicle
            [
                a owl:Restriction ;
                owl:hasValue true ;
                owl:onProperty :hasAirbags
            ]
        )
    ] .

:Truck a owl:Class .

:hasAirbags a owl:DatatypeProperty .



### 02.08.4.2 Restrictions: `some` / `only` / `onlysome` / `value` / `Self`, and cardinality

A restriction describes the individuals that have a certain kind of link through a property `p`:

- `p some C` is the individuals that have at least one `p` value in `C`.
- `p only C` is the individuals whose `p` values are all in `C`. It does not say there is any value, and a value that is stated is concluded to be in `C` and not rejected.
- `p onlysome C` is shorthand for `(p some C) and (p only C)` together.
- `p value i` is the individuals linked by `p` to the specific individual `i`.
- `p Self` is the individuals that are linked by `p` to themselves.
- `p min n C`, `p max n C` and `p exactly n C` are the individuals with at least, at most, or exactly `n` distinct `p` values in `C`. The class is optional: with no class, any value counts.

Restrictions are normally used after `SubClassOf:` (every such individual has this property) or `EquivalentTo:` (an individual is in the class exactly when it has it).

Below, one restriction of each kind is parsed and the RDF is printed. Compare `SmallCar` (qualified by a class) with `BigCar` (a count of any values).

In [4]:
_ = show("""
Class: Car
Class: Person
ObjectProperty: hasPassenger
ObjectProperty: hasOwner
Individual: Alice
ObjectProperty: monitorsItself
DataProperty: hasDoors

Class: FullCar
    SubClassOf: Car
    SubClassOf: hasPassenger some Person

Class: PassengerOnlyCar
    SubClassOf: Car
    SubClassOf: hasPassenger only Person

Class: BalancedCar
    SubClassOf: Car
    SubClassOf: hasPassenger onlysome Person

Class: AlicesCar
    SubClassOf: Car
    SubClassOf: hasOwner value Alice

Class: SelfMonitoringCar
    SubClassOf: Car
    SubClassOf: monitorsItself Self

Class: Sedan
    SubClassOf: Car
    SubClassOf: hasDoors exactly 4

Class: SmallCar
    SubClassOf: Car
    SubClassOf: hasPassenger max 2 Person

Class: BigCar
    SubClassOf: Car
    SubClassOf: hasPassenger min 5 Person
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Alice a owl:NamedIndividual .

:AlicesCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:hasValue :Alice ;
        owl:onProperty :hasOwner
    ] .

:BalancedCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Class ;
        owl:intersectionOf (
            [
                a owl:Restriction ;
                owl:onProperty :hasPassenger ;
                owl:someValuesFrom :Person
            ]
            [
                a owl:Restriction ;
                owl:allValuesFrom :Person ;
                owl:onProperty :hasPassenger
            ]
        )
    ] .

:BigCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:minQualifiedCardinality 5 ;
        owl:onClass :Person ;
        owl:onProperty :hasPassenger
    ] .

:Car a owl:Class .

:FullCar a owl:Class 

### 02.08.4.3 Enumerations: `{a, b, c}`

An enumeration is a class defined by listing its members (`owl:oneOf`). The class has exactly those individuals and no others. In the example, `PrimaryColor EquivalentTo: {Red, Green, Blue}` means a color is a `PrimaryColor` if and only if it is `Red`, `Green` or `Blue`.

Below, the enumeration is parsed and the RDF is printed.

In [5]:
_ = show("""
Individual: Red
Individual: Green
Individual: Blue

Class: PrimaryColor
    EquivalentTo: {Red, Green, Blue}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Blue a owl:NamedIndividual .

:Green a owl:NamedIndividual .

:PrimaryColor a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:oneOf ( :Red :Green :Blue )
    ] .

:Red a owl:NamedIndividual .



### 02.08.4.4 Data ranges and `Datatype:` frames

A data range is the set of data values a data property can have. It is the data-value counterpart of a class expression:

- `xsd:integer[>= 0, <= 400]` is a datatype restricted by facets: the integers from 0 to 400.
- `xsd:string or xsd:integer` is the union of two datatypes.
- `{"active", "retired"}` is an explicit list of the allowed values (`owl:oneOf` for data).
- A `Datatype:` frame gives a name to a data range: `PositiveInt EquivalentTo: xsd:integer[>= 0]`.

A value outside the range of its property makes the ontology inconsistent for a DL reasoner.

Below, each kind is parsed and the RDF is printed.

In [8]:
_ = show("""
Datatype: PositiveInt
    EquivalentTo: xsd:integer[>= 0]

DataProperty: hasSpeedKmh
    Range: xsd:integer[>= 0, <= 400]

DataProperty: hasFuelType
    Range: xsd:string or xsd:integer

DataProperty: hasStatusCode
    Range: {"active", "retired"}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:PositiveInt a rdfs:Datatype ;
    owl:equivalentClass [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions (
            [
                xsd:minInclusive 0
            ]
        )
    ] .

:hasFuelType a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:unionOf ( xsd:string xsd:integer )
    ] .

:hasSpeedKmh a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions (
            [
                xsd:minInclusive 0
            ]
            [
                xsd:maxInclusive 400
            ]
        )
    ] .

:hasStatusCode a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:oneOf ( "active" "retired" )
    ] .



## 02.08.5 Annotations

Annotations attach labels, comments and other notes to a frame, or to a single axiom inside a frame. The sections below show both forms.

Annotations are notes for people and tools. A reasoner ignores them: they do not change what the ontology means.

### 02.08.5.1 Frame-level `Annotations:`

An `Annotations:` clause on a frame is about the frame's own entity. It maps straight to plain triples with `rdfs:label`, `rdfs:comment`, or any other annotation property.

Below, a class is given a label and a comment and the RDF is printed.

In [10]:
_ = show("""
Class: Car
    Annotations: rdfs:label "Car", rdfs:comment "A wheeled motor vehicle"
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:comment "A wheeled motor vehicle" ;
    rdfs:label "Car" .



### 02.08.5.2 Inline per-axiom `Annotations:`

A more targeted form annotates one axiom and not the whole frame. An `Annotations:` block written inside another clause's list, immediately before one item, applies to just that axiom. The tree records this with an `owl:Axiom` node (`owl:annotatedSource`, `owl:annotatedProperty` and `owl:annotatedTarget`, plus the annotation itself), alongside the plain axiom triple and never instead of it. This is how to say who stated an axiom, or why.

Below, `Cat SubClassOf: Animal` carries a comment. `Pet` is also asserted as a superclass of `Cat`, but it is not annotated, because the annotation applies only to the item it precedes.

In [11]:
_ = show("""
Class: Animal
Class: Pet
Class: Cat
    SubClassOf: Annotations: rdfs:comment "domesticated" Animal, Pet
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Animal a owl:Class .

:Cat a owl:Class ;
    rdfs:subClassOf :Animal, :Pet .

:Pet a owl:Class .

_:b0 a owl:Axiom ;
    rdfs:comment "domesticated" ;
    owl:annotatedProperty rdfs:subClassOf ;
    owl:annotatedSource :Cat ;
    owl:annotatedTarget :Animal .



## 02.08.6 Top-level Misc axioms

Some axioms relate several entities at once and do not belong to one frame. Each accepts two or more names, not just a pair:

- `EquivalentClasses: A, B, C` means the classes have exactly the same members. `DisjointClasses: A, B, C` means no individual is in two of them.
- `EquivalentProperties: p, q` means the properties link the same pairs of individuals. `DisjointProperties: p, q` means no two individuals are linked by both.
- `SameIndividual: a, b` means the names refer to the same individual. `DifferentIndividuals: a, b` means they are all different individuals.

Below, each of these is parsed and the RDF is printed.

In [12]:
_ = show("""
Class: Auto
Class: Car
Class: Motorcar
EquivalentClasses: Auto, Car, Motorcar

Class: Motorcycle
Class: Bicycle
DisjointClasses: Car, Motorcycle, Bicycle

ObjectProperty: hasOwner
ObjectProperty: ownedBy
EquivalentProperties: hasOwner, ownedBy

ObjectProperty: hasPart
DisjointProperties: hasOwner, hasPart

Individual: mycar
Individual: myauto
SameIndividual: mycar, myauto

Individual: yourcar
DifferentIndividuals: mycar, yourcar
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Auto a owl:Class ;
    owl:equivalentClass :Car .

:Bicycle a owl:Class .

:Car a owl:Class ;
    owl:equivalentClass :Motorcar .

:Motorcar a owl:Class .

:Motorcycle a owl:Class .

:hasOwner a owl:ObjectProperty ;
    owl:equivalentProperty :ownedBy ;
    owl:propertyDisjointWith :hasPart .

:hasPart a owl:ObjectProperty .

:myauto a owl:NamedIndividual .

:mycar a owl:NamedIndividual ;
    owl:differentFrom :yourcar ;
    owl:sameAs :myauto .

:ownedBy a owl:ObjectProperty .

:yourcar a owl:NamedIndividual .

_:b0 a owl:AllDisjointClasses ;
    owl:members ( :Car :Motorcycle :Bicycle ) .



## 02.08.7 Known limitations

SWRL rules are not supported. The W3C Manchester Syntax has no rules. Some tools, such as Protégé and the OWL API, extend it with a `Rule:` frame for SWRL rules. StarLayer rejects a `Rule:` frame with a `ManchesterSyntaxError` that says SWRL rules are not supported by StarLayer, rather than silently dropping or mis-parsing it. For rules, see the [SRL guide](04.01-srl-sparql-rules.ipynb).

Annotations on the six top-level Misc axioms from section 02.08.6, `EquivalentClasses:` and friends, are supported. A one-member `EquivalentClasses:` is rejected, because the grammar needs two or more. Full conformance tracking, including what is matched, what is not, and why, lives in [manchester_syntax_gap_analysis.md](../../docs/design/graph/manchester_syntax_gap_analysis.md).

Below, a `Rule:` and a one-member `EquivalentClasses:` are each rejected with a `ManchesterSyntaxError`.

In [21]:
from starlayer.graph.manchester.errors import ManchesterSyntaxError

try:
    show("""
    Rule: Person(?x)
    """)
except ManchesterSyntaxError as e:
    print(f"Rejected as expected: {e.why}")

try:
    show("""
    Class: A
    Class: B
    EquivalentClasses: A
    """)
except ManchesterSyntaxError as e:
    print(f"Rejected as expected: {e.why}")


Rejected as expected: SWRL rules ('Rule:') are not supported by StarLayer: the W3C Manchester Syntax has no rules, they are an extension of some tools (Protege, the OWL API). Remove the rule, or use SRL (SPARQL Rules) for rules
Rejected as expected: EquivalentClasses: needs at least two members, got 1
